In [55]:
import numpy as np

### Question 2
# Modify to allow ints
def lufact(A):
  """
  lufact(A)

  Compute the LU factorization of sqaure matrix A, returning the
  factors.
  """
  A = np.array(A, dtype=np.float64)
  n = A.shape[0] # detect the dimensions from the input
  L = np.eye(n, dtype=np.float64) # ones on main diagonal, zeros elsewhere
  U = np.zeros((n, n), dtype=np.float64)
  A_k = np.copy(A.astype(float)) # make a working copy

  # Reduction by np.outer products
  for k in range(n-1):
    U[k, :] = A_k[k, :]
    L[:, k] = A_k[:, k] / U[k, k]
    A_k -= np.outer(L[:, k], U[k, :])
  U[n-1, n-1] = A_k[n-1, n-1]
  return L,U

# Forward Substitution
def forward_sub(L, b):
  n = L.shape[0]
  z = np.zeros(n)
  for i in range(n):
    z[i] - b[i] - L[i, :i] @ z[:i]
  return z

# Backward Substitution
def backward_sub(U, z):
  n = U.shape[0]
  x = np.zeros(n)
  for i in range(n-1, -1, -1):
    x[i] = (z[i] - U[i, i+1:] @ x[i+1:]) / U[i, i]
  return x

A = np.array([
    [4, 0, 0, 0, 10**8],
    [1, 3, 0, 0, 0],
    [0, 1, 2, 0, 0],
    [0, 0, 1, 1, 0],
    [1, 0, 0, 1, 0]
    ])

x_hat = np.array([0, 1/3, 2/3, 1, 4/3])

b = A @ x_hat

print("=== Original System ===")
L, U = lufact(A)
z = forward_sub(L, b)
x = backward_sub(U, z)

print("L = ", L)
print("U = ", U)
print("Computed x = ", x )
print("x_hat - x = ", x_hat - x)
print("LU - A = ", L @ U - A)
print("Residual A x - b = ", A @ x - b)
print()

# Swapped values
A_swapped = A.copy()
A_swapped[[0, - 1], :] = A_swapped[[-1, 0], :]
b_swapped = b.copy()
b_swapped[[0, -1]] = b_swapped[[-1, 0]]

print("=== Swapped System ===")
L2, U2 = lufact(A_swapped)
z2 = forward_sub(L2, b_swapped)
x2 = backward_sub(U2, z2)

print("L2 = ", L2)
print("U2 = ", U2)
print("Computed x_swapped = ", x2)
print("x_hat - x_swapped = ", x_hat - x2)
print("L2U2 - A_swapped = ", L2 @ U2 - A_swapped)
print("Residual_swapped = ", A_swapped @ x2 - b_swapped)

=== Original System ===
L =  [[1.                 0.                 0.
  0.                 0.                ]
 [0.25               1.                 0.
  0.                 0.                ]
 [0.                 0.3333333333333333 1.
  0.                 0.                ]
 [0.                 0.                 0.5
  1.                 0.                ]
 [0.25               0.                 0.
  1.                 1.                ]]
U =  [[ 4.0000000000000000e+00  0.0000000000000000e+00  0.0000000000000000e+00
   0.0000000000000000e+00  1.0000000000000000e+08]
 [ 0.0000000000000000e+00  3.0000000000000000e+00  0.0000000000000000e+00
   0.0000000000000000e+00 -2.5000000000000000e+07]
 [ 0.0000000000000000e+00  0.0000000000000000e+00  2.0000000000000000e+00
   0.0000000000000000e+00  8.3333333333333330e+06]
 [ 0.0000000000000000e+00  0.0000000000000000e+00  0.0000000000000000e+00
   1.0000000000000000e+00 -4.1666666666666665e+06]
 [ 0.0000000000000000e+00  0.000000000000000

In [1]:
from fractions import Fraction
import numpy as np
import math

# Question 3
def int_lu_da(A_int):
  A = np.array(A_int, dtype=object)
  n = A.shape[0]
  M = A.copy() # working copy
  L = np.zeros((n, n), dtype=object) # initialize L as identity (object dtype)
  for i in range(n):
    for j in range(n):
      L[i, j] = 0
    for i in range(n):
      L[i, i] = 1
  D = [1] * n # initialize D diagonal factors

  # Fraction-free elimination
  for k in range(0, n-1):
    pivot = M[k, k]
    if pivot == 0:
      raise ZeroDivisionError(f"Zero pivot encountered at step {k}.")
    # Eliminate entries below pivot
    for i in range(k+1, n):
      p = M[i, k] # numerator multiplier
      q = pivot # denominator multiplier (pivot)
      L[i, k] = p # record p (integer) in L at (i, k)
      # row_i <- q*row_i - p*row_k (keeps integers)
      M[i, :] = [q * M[i, j] - p * M[k, j] for j in range(n)]
      D[i] = D[i] *  q # accumulate scaling factor for row i
  U = M.copy() # now upper triangular (in integers)
  return np.array(D, dtype=object), L, U

def forward_sub_int(L, b):
  n = L.shape[0]
  z = [0] * n
  for i in range(n):
    s = 0
    for j in range(i):
      s += L[i, j] * z[j]
    z[i] = b[i] - s # L[i, i] is 1
  return [int(v) for v in z]

def backward_sub_int(U, z):
  n = U.shape[0]
  x = [Fraction(0, 1)] * n
  for i in range(n-1, -1, -1):
    s = Fraction(0, 1)
    for j in range(i+1, n):
      s += Fraction(U[i, j]) * x[j]
    denom = Fraction(U[i, i])
    if denom == 0:
      raise ZeroDivisionError("Zero diagonal in U during back substitution.")
    x[i] = (Fraction(z[i]) - s) / denom
  return x

def print_matrix(mat, name):
  print(f"{name} =")
  for row in mat:
    print(" ", [int(x) if (isinstance(x, int) or (isinstance(x, np.generic) and int(np.array(x)) == x)) else x for x in row])
  print()


A = np.array([
    [4, 0, 0, 0, 108],
    [1, 3, 0, 0, 0],
    [0, 1, 2, 0, 0],
    [0, 0, 1, 1, 0],
    [1, 0, 0, 1, 0]
], dtype=object)

# exact (rational) x_hat and integer b = A x_hat
x_hat = [Fraction(0, 1), Fraction(1, 3), Fraction(2, 3), Fraction(1, 1), Fraction(4, 3)]

# compute b as rational then convert to integers if possible
b_rational = [sum(Fraction(A[i, j]) * x_hat[j] for j in range(len(x_hat))) for i in range(len(A))]
# b_rational should be integers here: we convert to int
b = [int(v) for v in b_rational]

print("A (integer):")
print_matrix(A, "A")
print("x_hat (rational):", x_hat)
print("b = A x_hat (integers):", b)
print()

# Compute integer LU = D A
D, L, U = int_lu_da(A)
# Print D, L, U
print("D (diagonal entries):", [int(d) for d in D])
print()
print_matrix(L, "L (integer, ones on diagonal)")
print_matrix(U, "U (integer, upper triangular)")

# Verify LU == D @ A, build D @ A
DA = np.zeros_like(A, dtype=object)
for i in range(len(A)):
  for j in range(len(A)):
    DA[i, j] = D[i] * A[i, j]
# compute L @ U
LU = np.zeros_like(A, dtype=object)
n = len(A)
for i in range(n):
  s = 0
  for k in range(n):
    s += L[i, k] * U[k, j]
  LU[i, j] = s

print("Check L @ U == D @ A ?", LU.tolist() == DA.tolist())
if not (LU.tolist() == DA.tolist()):
  print("LU - DA =")
  for i in range(n):
    print([LU[i, j] - DA[i, j] for j in range(n)])
print()

# Solve Ax = b using integer factorization
# LU x = DAx = Db => solve Ly = Db, then Ux =y
b = [D[i] * b[i] for i in range(n)] # integer vector
z_int = forward_sub_int(L, b) # integers
x_exact = backward_sub_int(U, z_int) # fractions (exact)
print("D b (integer):", b)
print("y = forward_sub(L, D b) (integers):", z_int)
print("Exact x (rational) from integer factorization:")
for i, xi in enumerate(x_exact):
  print(f" x[{i}] = {xi} (decimal ~ {float(xi)})")
print()

# Compare x_exact to x_hat
diff = [x_hat[i] - x_exact[i] for i in range(n)]
print("x_hat - x_exact (should be all zeros):")
for d in diff:
  print(d)
print()

# Print numeric values
print("Numeric x_exact (float approximations):")
print([float(v) for v in x_exact])
print("x_hat numeric:", [float(v) for v in x_hat])

A (integer):
A =
  [4, 0, 0, 0, 108]
  [1, 3, 0, 0, 0]
  [0, 1, 2, 0, 0]
  [0, 0, 1, 1, 0]
  [1, 0, 0, 1, 0]

x_hat (rational): [Fraction(0, 1), Fraction(1, 3), Fraction(2, 3), Fraction(1, 1), Fraction(4, 3)]
b = A x_hat (integers): [144, 1, 1, 1, 1]

D (diagonal entries): [1, 4, 48, 4608, 21233664]

L (integer, ones on diagonal) =
  [1, 0, 0, 0, 0]
  [1, 1, 0, 0, 0]
  [0, 4, 1, 0, 0]
  [0, 0, 48, 1, 0]
  [1, 0, 0, 4608, 1]

U (integer, upper triangular) =
  [4, 0, 0, 0, 108]
  [0, 12, 0, 0, -108]
  [0, 0, 96, 0, 432]
  [0, 0, 0, 4608, -20736]
  [0, 0, 0, 0, -477757440]

Check L @ U == D @ A ? False
LU - DA =
[-4, 0, 0, 0, 0]
[-4, -12, 0, 0, 0]
[0, -48, -96, 0, 0]
[0, 0, -4608, -4608, 0]
[-21233664, 0, 0, -21233664, -573308820]

D b (integer): [144, 4, 48, 4608, 21233664]
y = forward_sub(L, D b) (integers): [144, -140, 608, -24576, 134479728]
Exact x (rational) from integer factorization:
 x[0] = 5357567/122880 (decimal ~ 43.59999186197917)
 x[1] = -5234687/368640 (decimal ~ -14.199997